# Community Observation IDs And Final Identity Groups

This notebook runs and inspects the identity step for the cumulative, interval, and overlapping snapshot approaches.

The pipeline now separates two concepts:

- Observation IDs: snapshot-scoped IDs such as `OVL-C0205`, where `02` is the snapshot and `05` is the community in that snapshot.
- Identity groups: event-linked chains of observations, later relabeled as final alive communities `c01`, `c02`, ... .

Identity inheritance uses prospective and retrospective stability with a default threshold of `0.4`. Mutual nomination is only used for concurrent split-and-merge cases.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "graph-matching":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "graph-matching"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from identity_events import run_approach

INPUT_DIR = NOTEBOOK_DIR / "outputs" / "graph_matching"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "stage_identification"
APPROACHES = ["cumulative", "interval", "overlap"]
JACCARD_THRESHOLD = 0.0
STABILITY_THRESHOLD = 0.4


## Generate Observation And Identity Tables


In [ ]:
for approach in APPROACHES:
    run_approach(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        approach=approach,
        jaccard_threshold=JACCARD_THRESHOLD,
        stability_threshold=STABILITY_THRESHOLD,
    )

print(f"Outputs written to {OUTPUT_DIR}")

## Summarize Identity Continuity

In [ ]:
summary_rows = []
for approach in APPROACHES:
    communities = pd.read_csv(OUTPUT_DIR / approach / "identified_communities.csv", keep_default_na=False)
    transitions = pd.read_csv(OUTPUT_DIR / approach / "identity_transitions.csv", keep_default_na=False)
    groups = pd.read_csv(OUTPUT_DIR / approach / "identity_groups.csv", keep_default_na=False)
    final_groups = pd.read_csv(OUTPUT_DIR / approach / "final_communities.csv", keep_default_na=False)

    summary_rows.append(
        {
            "approach": approach,
            "community_observations": len(communities),
            "observation_ids": communities["observation_id"].nunique(),
            "identity_groups": len(groups),
            "final_alive_communities": len(final_groups),
            "inherited_transitions": (transitions["decision"] == "inherited").sum(),
            "new_transitions": (transitions["decision"] == "new").sum(),
            "multi_snapshot_groups": (groups["lifespan_snapshots"].astype(int) > 1).sum(),
            "longest_lifespan_snapshots": groups["lifespan_snapshots"].astype(int).max(),
        }
    )

summary = pd.DataFrame(summary_rows)
summary


## Inspect Transition Decisions

The transition table preserves observation IDs, winning stability values, and candidate parent IDs so inheritance decisions remain auditable.


In [ ]:
approach_to_inspect = "interval"
transitions = pd.read_csv(
    OUTPUT_DIR / approach_to_inspect / "identity_transitions.csv",
    keep_default_na=False,
)
transitions.head(15)
